In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'torch', 'torchvision']
imports = {'pillow': 'PIL'}
pinned = {}
fallbacks = {'torch': 'torch==2.11.0', 'torchvision': 'torchvision==0.26.0'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'torch.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Modules and Model Construction

The multilayer perceptron of that section had four
parameter arrays, and we could name each one by hand. The models in the rest
of this book have many more. A residual network for images stacks dozens of
layers, and the language model Llama 3 8B
[@Grattafiori.Dubey.Jauhri.ea.2024], with 8 billion parameters,
consists of 32 identical blocks and stores its parameters in 291 arrays.
Every one of them must reach the optimizer, be written to a checkpoint, and
move to the GPU. An array that is left out raises no error: it is simply
never trained, or never saved.

Deep learning frameworks solve this with one abstraction, the *module*. A
module is an object that owns parameters, owns other modules (its
*children*), and defines a forward computation from inputs to outputs.
Layers, groups of layers, and whole models are all modules, so a model is a
tree of modules, as in the figure. The framework finds every
parameter by walking this tree, provided that each child has been
*registered* with its parent. This section is a practical one, and a reader
who has written modules before will move through much of it quickly. It
shows how registration works and how it fails, which is the part not to
skip, how to write the forward computation of a module ourselves, and how
to describe a whole model by a handful of numbers. The rest of the chapter uses the same walk for other tasks: counting
the parameters (that section), initializing them
(that section), saving them (that section), and
moving them to a GPU (that section).

![The module tree of the residual MLP that we build at the end of this section. Each edge carries the name under which the parent registered the child, and the path of names from the root to a parameter is the name of that parameter, here `1.body.0.weight`. Blocks 2 to 4 repeat block 1.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-module-names.svg)

In [ ]:
import math
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

In [ ]:
from dataclasses import dataclass

## The Module Abstraction

### Sequential Containers and the Registry

Every model since that section has been a module: `d2l.Module`
is a subclass of the framework's module class. The quickest way to build a
module from layers is a *sequential container*, a module that applies its
children one after the other. Here it holds the MLP of
that section, with a hidden layer of 256 ReLU units and
10 outputs, applied to a minibatch of two examples with 20 features each.

In PyTorch the module class is `nn.Module`, and the container
`nn.Sequential` takes its layers as arguments. Each `nn.Linear` receives
its number of inputs and of outputs.

In [ ]:
net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))

X = torch.rand(2, 20)
net(X).shape

The container keeps its children in a *registry*, a table from names to
modules, and its forward computation runs through that table in order.

`named_children` lists the registry. The names are the positions `'0'`,
`'1'`, and `'2'`; the ReLU is a module without parameters.

In [ ]:
dict(net.named_children())

This table is where every walk over the model starts. Collecting the
parameters for the optimizer, writing them to a file, and moving them to a
GPU all visit the registered children, then their children, and so on down
to the parameters. For all three, a layer that is not in the table does not
exist.

### Subclasses of the Module Class

A sequential container only expresses a chain. For any other wiring we
subclass the module class and write two methods: a constructor that creates
the children, and a forward method that applies them to the input. The
forward method is called `forward` in PyTorch and Gluon, `call` in Keras,
and `__call__` in NNX.


Here is the same MLP once more:

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.out = nn.Linear(256, 10)

    def forward(self, X):
        return self.out(F.relu(self.hidden(X)))

In [ ]:
net = MLP()
net(X).shape

Two details make this work. First, `self.hidden = ...` is not an ordinary
attribute assignment. The module class intercepts assignment (the Python
method `__setattr__`), and when the value is a module it enters the
registry under the attribute name, here `hidden` and `out`. That is all it
takes for both layers' parameters to reach the optimizer. Second, we wrote
no backward method: automatic differentiation derives the gradients from
whatever the forward method computes.

We call the model as `net(X)`, never as `net.forward(X)`:
`nn.Module.__call__` runs `forward` and, around it, any *hooks* registered
on the module, functions that see its inputs and outputs
(that section).

## Module Containers

### Writing a Container

A container has two jobs: to register each child, and to run the children
in order. It is instructive to write one ourselves, since the first job is
the one that can go wrong.

In [ ]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            self.add_module(str(idx), module)

    def forward(self, X):
        for module in self.children():
            X = module(X)
        return X

`add_module` enters a child into the registry under a name, here its
position as a string, and `children()` runs through the registry in the
order of insertion.

Our container works like the built-in one:

In [ ]:
net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net(X).shape

### Layers in a Plain List

Now suppose that we store the layers in an ordinary Python list, as anyone
would who had not read the previous subsection. The forward method can
still loop over the list, but the list itself is not a module. Does the
framework still find the layers inside it?

In [ ]:
class PlainListMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = [nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10)]

    def forward(self, X):
        for layer in self.layers:
            X = layer(X)
        return X

net = PlainListMLP()
net(X).shape, sum(p.numel() for p in net.parameters())

The model computes, yet it owns zero parameters. A plain Python list is not
a module, so `__setattr__` stores it like any other value and registers
nothing inside it. The layers still hold their own weights, which is why
the forward pass runs. But `net.parameters()` is empty, `net.to(device)`
leaves every layer on the CPU, and `net.state_dict()` saves an empty model.

How would we notice? Here the mistake is loud: `torch.optim.SGD` refuses an
empty list of parameters, so this model cannot even be handed to an
optimizer. The dangerous model is the one that is only partly registered.
In the model below the outer layers are attributes, and the two layers
between them sit in a plain list. We take one step of gradient descent and
ask which weights changed:

In [ ]:
class PartlyRegisteredMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.first = nn.Linear(20, 256)
        self.blocks = [nn.Linear(256, 256), nn.Linear(256, 256)]  # Plain list
        self.last = nn.Linear(256, 10)

    def forward(self, X):
        X = F.relu(self.first(X))
        for block in self.blocks:
            X = F.relu(block(X))
        return self.last(X)

In [ ]:
net = PartlyRegisteredMLP()
layers = [net.first, *net.blocks, net.last]
before = [layer.weight.clone() for layer in layers]
optimizer = torch.optim.SGD(net.parameters(), lr=0.1)
net(X).sum().backward()
optimizer.step()
[not torch.equal(b, layer.weight) for b, layer in zip(before, layers)]

Only the weights of the outer layers changed, and nothing failed.
Backpropagation passed through the two layers in the list to reach the
first one, but the optimizer never received their parameters. On the CPU
the only sign is a model that learns less than it should, which is the kind
of bug that survives for months. On a GPU the mistake is loud again:
`Trainer(num_gpus=1)` moves only the registered layers, and the first
forward pass stops with an error about tensors on different devices.

The registered container for a list of children is `nn.ModuleList`:

In [ ]:
class ModuleListMLP(PlainListMLP):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList(self.layers)

net = ModuleListMLP()
net(X).shape, sum(p.numel() for p in net.parameters())

Same forward pass, and now all 7,946 parameters are registered: 20 · 256 +
256 in the hidden layer and 256 · 10 + 10 in the output layer.

Which container, then? `nn.Sequential` registers its children and supplies
a forward method that chains them. `nn.ModuleList` registers a list of
children and supplies no forward method, so we write the loop ourselves,
which is what we want when the loop is not a plain chain: blocks that take
an extra argument, or a computation between blocks. `nn.ModuleDict` does
the same for children indexed by name. A model that repeats one block many
times, such as the 32-block 8B model of the introduction, keeps the blocks
in an `nn.ModuleList`.

## Forward Computation

The forward method is ordinary Python. It can call any tensor operation,
loop, branch, and combine intermediate results. We use this freedom for a
residual connection, and then ask what a compiled training step takes away
from it.

### Residual Connections

Let us start with the simplest module whose forward method is not a chain.
A *residual connection* adds the input of a block to its output
[@He.Zhang.Ren.ea.2016*1]. If $f$ is the *body* of the block, the
computation inside it, here two fully connected layers with a ReLU between
them, the block computes

$$\mathbf{y} = \mathbf{x} + f(\mathbf{x}).$$

If $f$ is zero, the block is the identity, so a stack of such blocks can
pass its input through unchanged, and each block only has to learn a
correction $f(\mathbf{x})$ to its input. The networks of
that section are built this way. No framework provides the residual
connection as a layer, and none needs to. It is one line of the forward
method, `X + self.body(X)`:

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, num_hiddens):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(num_hiddens, num_hiddens), nn.ReLU(),
            nn.Linear(num_hiddens, num_hiddens))

    def forward(self, X):
        return X + self.body(X)

![A residual block. The input splits into the body, Linear, ReLU, Linear, and an identity path, and the two rejoin by addition.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-residual-block.svg)

the figure shows the wiring. The addition has one
consequence for construction: $\mathbf{x}$ and $f(\mathbf{x})$ must have
the same shape, so a residual block has a single width, which we pass to
the constructor.

In [ ]:
block = ResidualBlock(24)
block(torch.randn(2, 24)).shape

A forward method may also read values that are neither inputs nor
parameters. A fixed number can stay a plain Python attribute. An array
computed from data, such as the mean of the inputs, is another matter: it
must be saved and moved with the model, yet never trained.
that section shows how to register such state.

### Eager and Compiled Execution

So far the forward method has also run like ordinary Python, one line at a
time. A framework can instead record the computation once and replay the
recording. This is faster, but it restricts what the forward method may do.

PyTorch runs `forward` as written, one operation at a time, a mode called
*eager execution*, and so does our `Trainer`. A loop whose condition depends
on the values of tensors therefore works just as it does in NumPy.
Compilers for PyTorch models exist, and that section uses
them.

## Lazy Initialization

### Shapes from Data

Since that section, our PyTorch models have used lazy
layers such as `nn.LazyLinear(256)`, which name only their output width.
The weight has shape `(256, in_features)`, and we never said what
`in_features` is: the layer reads it from the first minibatch. Here is the
mechanism. Before the first forward pass the weight is a placeholder
without a shape:

In [ ]:
net = nn.Sequential(nn.LazyLinear(256), nn.ReLU(), nn.LazyLinear(10))
net[0].weight

The first forward pass reads the input width from the minibatch, creates
the weight, and turns the layer into an ordinary `nn.Linear`. The output
width of each layer is the input width of the next, so the shapes cascade
through the model:

In [ ]:
net(X)
net[0].weight.shape

### Initialization after a Dry Run

Lazy layers spare us the arithmetic of input widths, which in a deep model
is tedious and easy to get wrong. Unfortunately, the parameters then do not
exist until a minibatch has passed through. Before that, they cannot be
counted, and an initializer whose scale depends on the number of inputs,
such as the Xavier initialization of that section, cannot be
applied. Initialization therefore follows a *dry run*, one forward pass on a
representative minibatch whose output is discarded.

The `Trainer` has done this before every training run since
that section, whenever the model still had lazy parameters:
a dry run, then the model's `weight_init`. The method behind it,
`apply_init` of `d2l.Module`, we have not shown so far. It runs the model
once without recording gradients, then calls the given function on every
submodule:

In [ ]:
@d2l.add_to_class(d2l.Module)
def apply_init(self, inputs, init=None):
    """Run a dry pass to fix lazy shapes, then apply `init`."""
    with torch.no_grad():
        self(*inputs)
    if init is not None:
        self.apply(init)

`nn.Module.apply(fn)` calls `fn` on every module of the tree, children
before their parents: one more task that is a walk over the tree. Here is
`apply_init` with Xavier's uniform initializer:

In [ ]:
class TinyMLP(d2l.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.LazyLinear(256), nn.ReLU(),
                                 nn.LazyLinear(10))

def init_xavier(module):
    if isinstance(module, nn.Linear):
        nn.init.xavier_uniform_(module.weight)

model = TinyMLP()
model.apply_init([X], init_xavier)
model.net[0].weight.shape

The dry run turned every lazy layer into an `nn.Linear`, after which
`init_xavier` could recognize the layers and redraw their weights.

Which should we prefer? For a quick experiment on new data, lazy layers are
a convenience. A model whose parameters are to be counted, initialized by a
rule, or built from a config should state both widths of every layer, as
the models in the rest of this chapter do.

## Models from a Config

### Configs and Builders

Could we rebuild a model from its saved weights alone? Not quite. A trained
model is a set of arrays plus the code that knows what to do with them.
Every width and depth in this section so far was a literal typed into a
constructor call, and a checkpoint of the weights records such choices at
best indirectly: the shapes of the arrays reveal the widths, but nothing in
them reveals a choice without parameters, such as the activation function.

The choices that vary from one experiment to the next therefore belong in a
small *configuration* object, or *config*, that is logged with the results
and saved with the weights. The topology, meaning what a block computes and
how blocks connect, stays in code. Here a Python `dataclass` holds four
fields, and `build` turns them into a model:

In [ ]:
@dataclass
class MLPConfig:
    num_inputs: int = 784
    num_hiddens: int = 256
    num_blocks: int = 4
    num_outputs: int = 10

def build(cfg: MLPConfig) -> nn.Module:
    blocks = [ResidualBlock(cfg.num_hiddens) for _ in range(cfg.num_blocks)]
    return nn.Sequential(nn.Linear(cfg.num_inputs, cfg.num_hiddens),
                         *blocks, nn.Linear(cfg.num_hiddens, cfg.num_outputs))

One config gives one architecture: an input layer from `num_inputs`
features to the hidden width, `num_blocks` residual blocks, and an output
layer. The blocks pass through a plain Python list, of all things. This is
safe here, because unpacking the list into the sequential container
registers each block. Every layer receives both of its widths, so the model
needs no dry run before its parameters can be counted.

The names of the first parameters are the paths of the figure,
joined by dots:

In [ ]:
net = build(MLPConfig())
[(name, tuple(p.shape)) for name, p in net.named_parameters()][:4]

The model maps a minibatch of 784-dimensional inputs, the size of a
flattened Fashion-MNIST image, to 10 outputs:

In [ ]:
net(torch.rand(2, 784)).shape

### Counting Parameters

The config also tells us the size of the model before we build it. The
input layer has $784 \cdot 256 + 256 = 200{,}960$ parameters, each block
has two layers of $256 \cdot 256 + 256$, or $131{,}584$ together, and the
output layer has $256 \cdot 10 + 10 = 2{,}570$. The default model therefore
has

$$200{,}960 + 4 \cdot 131{,}584 + 2{,}570 = 729{,}866$$

parameters, most of them in the blocks. Rescaling the model is a change to
two fields, not an edit of the model code, and a walk over the parameters
confirms the count:

In [ ]:
for cfg in (MLPConfig(), MLPConfig(num_hiddens=512, num_blocks=8)):
    n = sum(p.numel() for p in build(cfg).parameters())
    print(f'num_hiddens={cfg.num_hiddens}, num_blocks={cfg.num_blocks}: '
          f'{n:,} parameters')

Doubling the width multiplies the parameters of each block by about four,
and doubling the number of blocks doubles their total, so the larger
config has more than six times as many parameters. The 8B model of the
introduction is described the same way: a config of a dozen numbers, such
as 32 blocks and a width of 4,096, and code that turns it into a module
tree.

Note, however, that a config records numbers, not code. Rebuilding a model
from a checkpoint needs the weights, the config, and the same `build`
function and block classes. that section saves the first two
together in one file; the code we must keep ourselves.


Throughout this section the parameters were things to be found and
counted. We have not yet looked at them: which tensors a model holds, how
much memory they take, and how to share or freeze them. This is the subject
of that section.

## Summary

Three things are worth remembering. A model is a tree of modules, and the
name of each parameter is its path from the root. The framework finds
parameters by walking the registered children, so a child stored where the
walk cannot see it is never trained or saved, and in the worst case nothing
tells us. A config of a few numbers, together with a `build` function,
records an architecture so that it can be logged, saved, and rebuilt. The
rest is technique: a forward method written in ordinary Python, which a
compiled training step restricts, and lazy layers, which must see a
minibatch before they can be initialized.

In PyTorch, children are registered by attribute assignment and by the
containers `nn.Sequential`, `nn.ModuleList`, and `nn.ModuleDict`. Modules in
a plain list stay unregistered, without any warning.

## Exercises

1. [code] **Unregistered modules.** The plain list in `PlainListMLP` keeps
   its layers out of `net.parameters()`, `net.to(device)`, and
   `net.state_dict()`. Predict, then verify, what three further operations
   do to the model.
    1. Call `net.apply(init_xavier)` and check whether the weight of
       `net.layers[0]` changes.
    1. Add `nn.Dropout(0.5)` after the `nn.ReLU()` in the list, call
       `net.eval()`, and pass the same input through the model twice.
    1. The method `load_state_dict` is the inverse of `state_dict`: it
       copies a dictionary of saved tensors into a model. Call
       `net.load_state_dict(ModuleListMLP().state_dict())`.
    1. Which of these failures are silent and which raise an error? Explain
       all three from the same missing registration.

2. **Widening a residual block.** `ResidualBlock` requires its input and
   output widths to agree. Suppose a block maps width $d$ to a larger width
   $d'$, so its body ends in a $d'$-wide layer and the skip path must also
   produce width $d'$. Give one way to build that skip path with
   parameters and one without. For each, count the extra parameters and
   state which output coordinates still receive the identity path.

3. [code] **Config boundaries.** Extend `MLPConfig` with a field
   `act: str = 'relu'` that selects the activation inside every
   `ResidualBlock`, and make `build` honor it.
    1. Build one model with `act='tanh'` and one from the default config,
       and compare their parameter shapes. Could a checkpoint that holds
       only the weights tell the two models apart? State what the answer
       implies for the config.
    1. A plain feed-forward block has the same `body` as `ResidualBlock`
       but no skip connection. Apply the same test to the choice between
       the two block types, and decide whether that choice belongs in the
       config or in code.

4. [code] **A module from scratch.** ● In plain Python and NumPy, without any
   framework base class, write a `Parameter` class that wraps an array and
   a minimal `Module` class whose `__setattr__` registers every assigned
   `Parameter` or `Module`. Give `Module` a `parameters()` method that
   recurses into the registered children. Write a `Linear` subclass of
   `Module` that holds a weight and a bias, build a two-layer MLP from two
   instances of it, and verify that `parameters()` returns all four arrays.
   Then store the two layers in a plain list, check what `parameters()`
   returns, and extend `Module` so that the list version finds all four
   arrays as well.

    *Adapted from CMU 10-714,
    [Homework 2](https://github.com/dlsyscourse/hw2), Question 2.*

5. [code] **Lazy input width.** Build
   `net = nn.Sequential(nn.LazyLinear(256), nn.ReLU(), nn.LazyLinear(10))`
   and pass a batch with 20 features through it, then a batch with 30
   features. Explain the outcome from what the first forward pass did to
   each layer. Then describe a model that accepts inputs of either width
   while sharing every layer after the first.

[Discussions](https://d2l.discourse.group/t/36752)